## 9. 🆕 Escenario propio de natalidad (RENAPER + DEIS + INDEC)

**Problema.** Para las cohortes 2025 en adelante, la serie principal usa los nacimientos proyectados por la ONU (WPP julio 2024): 508 mil en 2025, un **rebote artificial**. La ONU proyectó 2023 y 2024 *antes* de tener el dato: estimó 504 mil y 507 mil, y los nacimientos reales (DEIS) fueron **460.902** y **413.135**. Los datos recientes del RENAPER (2025 provisorio) y de la TGF indican que la caída continúa.

**Qué hace esta sección** (fragmento autónomo; depende solo de `d`, `s`, `TAU`, `RHO`, `DATA`, `OUT` en memoria):

1. arma un dato base 2025 **estimado** (RENAPER ajustado por la relación RENAPER/DEIS);
2. proyecta nacimientos 2025–2035 con tres escenarios de TGF **propuestos para que el grupo los valide**;
3. exporta `natalidad_escenarios_2025_2035.csv`;
4. recalcula $TBP_C$ y $TBP_A$ para las cohortes 2025–2035 y los compara con la serie actual (N de la ONU).

> **Aviso 1.** Los valores del RENAPER son los de la **descarga CSV del tablero** (nacimientos identificados y TGF, total país). El de **2025 es provisorio** y no equivale a los nacimientos *ocurridos* del DEIS.
> **Aviso 2.** $J$ (población 65+) **sigue siendo el de la ONU y NO está re-ponderado todavía**: eso se hace en la sección 10. Mientras tanto, el nivel de $TBP$ de las cohortes ≥2025 es parcial.

### 9.1 Carga de datos

Se cargan dos archivos de `datos/procesados/`: los datos del RENAPER y la población por edad simple de la ONU (mujeres, variante media). Rutas candidatas: `../datos/procesados`, `datos/procesados`, `DATA` y `datos_tbp`; si no están, se descargan del repositorio (URL *raw*) a la carpeta de salidas.

In [ ]:
import os, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print

RAW_BASE_9 = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
CANDS_9 = ["../datos/procesados", "datos/procesados", DATA, "datos_tbp", "../datos_tbp"]

def buscar_archivo_9(nombre):
    for c in CANDS_9:
        p = os.path.join(c, nombre)
        if os.path.exists(p):
            return p
    os.makedirs(OUT, exist_ok=True)          # fallback: descarga desde GitHub (raw)
    p = os.path.join(OUT, nombre)
    urllib.request.urlretrieve(f"{RAW_BASE_9}/{nombre}", p)
    return p

f_ren = buscar_archivo_9("renaper_natalidad_2012_2025.csv")
f_pob = buscar_archivo_9("un_wpp2024_argentina_poblacion_edad_simple.csv")
print("RENAPER :", os.path.abspath(f_ren)); print("Poblacion ONU:", os.path.abspath(f_pob))

ren = pd.read_csv(f_ren).set_index("anio")
ren["nac"] = ren["nacimientos_identificados_miles"] * 1000      # personas
pob = pd.read_csv(f_pob, low_memory=False)
if "Variant" in pob.columns:
    pob = pob[pob["Variant"] == "Medium"]
pob["edad"] = pd.to_numeric(pob["AgeGrpStart"], errors="coerce")
# M_t: mujeres de 15 a 49 anios en t (ONU medio), en miles
M = pob[(pob.edad >= 15) & (pob.edad <= 49)].groupby("Time")["PopFemale"].sum()

deis = s["deis_nacimientos"]; onu = s["un_births_miles"] * 1000
display(pd.DataFrame({"DEIS": deis.loc[2019:2024], "RENAPER": ren.nac.loc[2019:2024],
                      "ONU": onu.loc[2019:2024]}).round(0))
print("TGF RENAPER:", ren.tgf_hijos_por_mujer.loc[2022:2025].to_dict())
print("Mujeres 15-49 (miles, ONU medio):", M.loc[[2025, 2030, 2035]].round(0).to_dict())

### 9.2 Dato base 2025: ESTIMACIÓN a partir de un dato provisorio

Los nacimientos del RENAPER son *personas identificadas* y no coinciden exactamente con los *nacimientos ocurridos* (DEIS). Para llevar el dato RENAPER a la escala DEIS (la que usa la serie hasta 2024) se lo divide por la **relación RENAPER/DEIS promedio 2022–2024**:

$$N_{2025}^{est} = \frac{N_{2025}^{RENAPER}}{\overline{\left(N^{RENAPER}/N^{DEIS}\right)}_{2022\text{–}2024}}$$

**Por qué esa ventana:** la relación es estable (0,98–1,05 en 2012–2024) y los años recientes son los más comparables con 2025 (misma cobertura de DNI digital). Es una **ESTIMACIÓN**, **no un dato oficial**: combina un valor provisorio con una lectura a ojo de un gráfico.

In [ ]:
# ---- Parametros del dato base (modificables) ----
VENTANA_RATIO = (2022, 2024)        # anios usados para la relacion RENAPER/DEIS
ANIO_BASE = 2025

ratio = (ren.nac / deis).loc[2012:2024]
r_med = ratio.loc[VENTANA_RATIO[0]:VENTANA_RATIO[1]].mean()
N_REN_2025 = ren.nac[ANIO_BASE]
N_2025_EST = N_REN_2025 / r_med
TGF_2025_OBS = ren.tgf_hijos_por_mujer[ANIO_BASE]

print("Relacion RENAPER/DEIS 2012-2024:"); print(ratio.round(3).to_string())
print(f"\nMedia {VENTANA_RATIO[0]}-{VENTANA_RATIO[1]}: {r_med:.3f}")
print(f"RENAPER 2025 (provisorio): {N_REN_2025:,.0f}")
print(f"N_2025 ESTIMADO = {N_2025_EST:,.0f}  <- ESTIMACION a partir de un dato provisorio, NO dato oficial")
print(f"(ONU proyecta {onu[2025]:,.0f} para 2025: {onu[2025]/N_2025_EST - 1:+.0%} respecto de la estimacion)")

### 9.3 Escenarios 2025–2035 (propuesta a validar por el grupo)

**Método.** Los nacimientos de un año son (TGF × mujeres en edad fértil) por una constante que convierte "TGF" a "nacimientos":

$$N_t = TGF_t \times K \times \frac{M_t}{M_{2025}},\qquad K = \frac{N_{2025}^{est}}{TGF_{2025}}$$

* $M_t$ = mujeres de 15–49 años en $t$ (ONU medio, `PopFemale`). Capta el *efecto de composición* (cuántas mujeres hay en edad de tener hijos): crece ~3% entre 2025 y 2035.
* $K$ calibra el nivel en 2025, de modo que los tres escenarios **coinciden en 2025** y difieren solo por la trayectoria de la TGF. Es una aproximación: supone que la estructura de edades de la fecundidad no cambia (la TGF real usa tasas por edad).

**Los tres escenarios** (parámetros editables en la celda siguiente):

* **(a) Persistencia:** la TGF se queda en 1,04 (nivel observado 2025). Referencia "nada cambia".
* **(b) Recuperación lenta:** sube linealmente de 1,04 (2025) a 1,32 (2035), que es el valor de la **variante media de INDEC para 2035** (INDEC, proyecciones 2022–2040, pp. 26–30). Es el escenario "optimista".
* **(c) Caída adicional:** baja a 0,95 en 2028 y se estabiliza. Es una cota inferior ilustrativa, no una predicción.

**Contexto INDEC.** Las proyecciones de INDEC usaron nacimientos DEIS solo hasta 2023 y suponen una TGF media de 1,27 (2025), 1,24 (2030) y 1,32 (2035). La TGF observada 2025 del RENAPER (~1,04) queda **por debajo incluso del mínimo de la variante baja de INDEC (1,17)**. Por eso INDEC aparece solo como TGF de referencia (no se calculan nacimientos con la TGF INDEC: su denominador de mujeres y su calibración son otros).

In [ ]:
# ============ PARAMETROS DE LOS ESCENARIOS (modificables) ============
TGF_2025 = TGF_2025_OBS         # 1,04: TGF RENAPER 2025 (provisorio)
TGF_REC_2035 = 1.32             # (b) valor INDEC variante media 2035
TGF_CAIDA_MIN = 0.95            # (c) piso de la caida adicional
ANIO_CAIDA = 2028               # (c) anio en que se alcanza el piso
ANIOS = list(range(2025, 2036))
A_LEGAL_9, A_OBS_9 = 30.0, 14.2  # 30 = requisito legal de referencia (NO es un maximo); 14,2 = observado ANSES
# Referencia INDEC (proyecciones 2022-2040, pp. 26-30, ya verificado)
INDEC_MEDIA = {2025: 1.27, 2030: 1.24, 2035: 1.32, 2040: 1.40}
INDEC_MIN_BAJA = {2025: 1.28, 2030: 1.22, 2035: 1.17}     # minimos citados de la variante baja
# =====================================================================

K = N_2025_EST / TGF_2025

def tgf_escenario(nombre, t):
    if nombre == "persistencia":
        return TGF_2025
    if nombre == "recuperacion_lenta":
        return TGF_2025 + (TGF_REC_2035 - TGF_2025) * (t - ANIOS[0]) / (ANIOS[-1] - ANIOS[0])
    if nombre == "caida_adicional":
        return TGF_2025 + (TGF_CAIDA_MIN - TGF_2025) * min(t - ANIOS[0], ANIO_CAIDA - ANIOS[0]) / (ANIO_CAIDA - ANIOS[0])
    raise ValueError(nombre)

ESCENARIOS = ["persistencia", "recuperacion_lenta", "caida_adicional"]
filas = []
for e in ESCENARIOS:
    for t in ANIOS:
        tg = tgf_escenario(e, t)
        filas.append(dict(anio=t, escenario=e, tgf=tg, nacimientos=tg * K * M[t] / M[ANIO_BASE]))
# referencias: ONU (nacimientos) e INDEC media (solo TGF, interpolada linealmente entre anios publicados)
for t in ANIOS:
    filas.append(dict(anio=t, escenario="onu_medio", tgf=np.nan, nacimientos=onu[t]))
ks = sorted(INDEC_MEDIA)
for t in ANIOS:
    filas.append(dict(anio=t, escenario="indec_media_tgf", tgf=np.interp(t, ks, [INDEC_MEDIA[k] for k in ks]), nacimientos=np.nan))
esc = pd.DataFrame(filas)

piv = esc.pivot(index="anio", columns="escenario", values="nacimientos")[ESCENARIOS + ["onu_medio"]]
display((piv / 1000).round(1).loc[[2025, 2026, 2028, 2030, 2035]].rename_axis("miles de nacimientos"))
print(f"TGF observada 2025 = {TGF_2025:.2f} < minimo variante baja INDEC 2035 = {INDEC_MIN_BAJA[2035]:.2f} y 2025 = {INDEC_MIN_BAJA[2025]:.2f}")

### 9.4 Salida: `natalidad_escenarios_2025_2035.csv`

Formato largo: `anio, escenario, tgf, nacimientos`. Incluye los tres escenarios y dos referencias: `onu_medio` (solo nacimientos; la TGF de la ONU no se cargó acá) e `indec_media_tgf` (solo TGF, interpolada linealmente entre 2025, 2030 y 2035; **nacimientos vacíos a propósito**: no se tiene la serie de nacimientos de INDEC y calcularla con otra metodología mezclaría supuestos).

In [ ]:
DEST = next((p for p in ["../datos/procesados", "datos/procesados"] if os.path.isdir(p)), OUT)
os.makedirs(DEST, exist_ok=True)
f_out = os.path.join(DEST, "natalidad_escenarios_2025_2035.csv")
esc.round({"tgf": 4, "nacimientos": 0}).to_csv(f_out, index=False)
print("Escrito:", os.path.abspath(f_out), esc.shape)

### 9.5 Recalcular el TBP con cada escenario

Se reemplaza $N_t$ por el del escenario **solo para las cohortes ≥2025** (cohortes ≤2024 quedan idénticas: DEIS). El resto de los insumos es el de `d`: $S_{65}$, $E_{65}$ y $J$.

$$TBP_C=\frac{N(1-\mu)\bar A\tau}{J\rho},\qquad TBP_A=\frac{TBP_C}{E_r}$$

> **J NO está re-ponderado todavía** (sigue siendo la población 65+ de la ONU, que descansa en los nacimientos ONU anteriores). Con menos nacimientos, la población 65+ de 2090–2100 que proyecta la ONU también sería menor, así que el denominador está sobreestimado y el efecto aquí es una **cota parcial**. La corrección va en la sección 10.

In [ ]:
base9 = d.set_index("cohorte")[["N_t", "S65", "J", "E65", "TBP_C_legal", "TBP_A_legal", "TBP_C_obs", "TBP_A_obs"]].copy()
N_snapshot = base9["N_t"].copy()                      # para el control "no cambia <=2024"

def serie_N(nombre):
    # N por cohorte: DEIS hasta 2024 (igual que d); desde 2025 el escenario (o la ONU, serie actual)
    N = base9["N_t"].copy()
    if nombre != "actual_onu":
        N.loc[ANIOS] = esc[esc.escenario == nombre].set_index("anio")["nacimientos"].loc[ANIOS].values
    return N

def tbp_9(N, A, tipo):
    c = N * base9.S65 * A * TAU / (base9.J * RHO)
    return c if tipo == "C" else c / base9.E65

NOMBRES = ["actual_onu"] + ESCENARIOS
N9 = {n: serie_N(n) for n in NOMBRES}
T9 = {(n, A, tipo): tbp_9(N9[n], A, tipo) for n in NOMBRES for A in (A_LEGAL_9, A_OBS_9) for tipo in "CA"}

# la serie "actual" reproduce la de d (control de consistencia)
assert np.allclose(T9[("actual_onu", A_LEGAL_9, "C")], base9.TBP_C_legal)
assert np.allclose(T9[("actual_onu", A_OBS_9, "A")], base9.TBP_A_obs)
print("OK: la serie 'actual_onu' recalculada reproduce TBP_C/TBP_A de d")

### 9.6 Gráficos

(i) Nacimientos 2012–2035: DEIS, RENAPER (sin ajustar), ONU y los tres escenarios (el punto 2025 común es la **estimación**). (ii) $TBP_C$ por cohorte 2000–2035 para $\bar A=30$ y $\bar A=14{,}2$, con los tres escenarios y la serie actual; la línea vertical marca el corte 2024/2025. **$\bar A=30$ es el requisito legal de referencia, no un máximo.** En ambos paneles de $TBP_C$, $J$ es el de la ONU (no re-ponderado).

In [ ]:
COL9 = {"persistencia": "#d55e00", "recuperacion_lenta": "#009e73", "caida_adicional": "#cc79a7", "actual_onu": "#444444"}
ETQ9 = {"persistencia": "(a) Persistencia (TGF 1,04)", "recuperacion_lenta": "(b) Recuperación lenta (a 1,32)",
        "caida_adicional": "(c) Caída adicional (a 0,95)", "actual_onu": "Serie actual (N de la ONU)"}

fig, ax = plt.subplots(1, 3, figsize=(20, 5.4))
a = ax[0]
a.plot(deis.loc[2012:2024].index, deis.loc[2012:2024] / 1e3, color="#0072b2", lw=2, label="DEIS (observado)")
a.plot(ren.index, ren.nac / 1e3, "s--", color="#e69f00", ms=4, lw=1.2, label="RENAPER (2025 prov.)")
a.plot(onu.loc[2012:2035].index, onu.loc[2012:2035] / 1e3, color="#444444", ls=":", lw=2, label="ONU WPP 2024 (medio)")
for e in ESCENARIOS:
    p = piv[e]; a.plot(p.index, p / 1e3, color=COL9[e], lw=2, label=ETQ9[e])
a.plot([2025], [N_2025_EST / 1e3], "o", color="k", ms=6, label=f"N 2025 estimado ({N_2025_EST/1e3:.0f} mil)")
a.axvline(2024.5, color="gray", ls="--", lw=1); a.set_title("(i) Nacimientos 2012–2035 (miles)")
a.set_xlabel("año"); a.grid(alpha=.3); a.legend(fontsize=7.5)

for a, AA, tit in [(ax[1], A_LEGAL_9, "Ā = 30 (requisito legal de referencia, no un máximo)"),
                   (ax[2], A_OBS_9, "Ā = 14,2 (observado ANSES, sesgado a la baja)")]:
    for n in NOMBRES:
        x = T9[(n, AA, "C")].loc[2000:2035]
        a.plot(x.index, x, color=COL9[n], lw=2.4 if n == "actual_onu" else 1.8, ls=":" if n == "actual_onu" else "-", label=ETQ9[n])
    a.axvline(2024.5, color="gray", ls="--", lw=1)
    a.text(2024.8, a.get_ylim()[1] * 0.98, "2024/2025", fontsize=8, color="gray", va="top")
    a.set_title(f"(ii) TBP_C por cohorte, {tit}", fontsize=9.5); a.set_ylabel("TBP_C (años)")
    a.set_xlabel("año de nacimiento de la cohorte"); a.grid(alpha=.3); a.legend(fontsize=7.5, loc="upper right")
fig.suptitle("Escenarios propios de natalidad 2025–2035 — J todavía NO re-ponderado (sección 10)", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.95])
os.makedirs(OUT, exist_ok=True)
plt.savefig(f"{OUT}/seccion_09_natalidad.png", dpi=130); plt.show()

### 9.7 Tabla resumen: cohortes 2025, 2030 y 2035

Nacimientos y $TBP$ por escenario frente a la serie actual (N de la ONU). Columna `Δ vs actual` = variación porcentual del $TBP_C$ respecto de la serie actual. **$J$ de la ONU, no re-ponderado.**

In [ ]:
COHS = [2025, 2030, 2035]
res = []
for n in NOMBRES:
    for c in COHS:
        r = dict(cohorte=c, escenario=n, N=N9[n][c])
        for A, lab in [(A_LEGAL_9, "A30"), (A_OBS_9, "A14,2")]:
            r[f"TBP_C ({lab})"] = T9[(n, A, "C")][c]
            r[f"TBP_A ({lab})"] = T9[(n, A, "A")][c]
        r["Δ vs actual (%)"] = 100 * (T9[(n, A_LEGAL_9, "C")][c] / T9[("actual_onu", A_LEGAL_9, "C")][c] - 1)
        res.append(r)
resumen = pd.DataFrame(res).sort_values(["cohorte", "escenario"]).set_index(["cohorte", "escenario"])
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
display(resumen.round({"N": 0, "Δ vs actual (%)": 1}))
print("Nota: J = poblacion 65+ de la ONU (NO re-ponderado; se hace en la seccion 10). Resultados parciales.")

### 9.8 Controles (asserts)

1. Relación RENAPER/DEIS en [0,95; 1,06] para 2012–2024 (se excluye 2020, año de pandemia).
2. Nacimientos de todos los escenarios positivos.
3. Los tres escenarios coinciden en 2025 (y valen $N_{2025}^{est}$).
4. El ajuste no cambia las cohortes ≤2024 (ni `N_t` ni el TBP).

In [ ]:
r_chk = ratio.drop(2020)
assert r_chk.between(0.95, 1.06).all(), r_chk[~r_chk.between(0.95, 1.06)]
print(f"OK 1) RENAPER/DEIS en [0,95; 1,06]: min {r_chk.min():.3f} | max {r_chk.max():.3f} (2012-2024 sin 2020; 2020 = {ratio[2020]:.3f})")

for e in ESCENARIOS:
    assert (esc[esc.escenario == e].nacimientos > 0).all()
print("OK 2) nacimientos > 0 en los 3 escenarios, 2025-2035")

n25 = esc[(esc.anio == 2025) & esc.escenario.isin(ESCENARIOS)].nacimientos
assert np.allclose(n25, N_2025_EST) and np.ptp(n25) < 1e-6
print(f"OK 3) los 3 escenarios coinciden en 2025: {n25.iloc[0]:,.0f}")

ant = base9.index <= 2024
for n in ESCENARIOS:
    assert np.array_equal(N9[n][ant].values, N_snapshot[ant].values)
    assert np.allclose(T9[(n, A_LEGAL_9, "C")][ant], base9.TBP_C_legal[ant])
    assert np.allclose(T9[(n, A_OBS_9, "A")][ant], base9.TBP_A_obs[ant])
assert np.array_equal(d.set_index("cohorte")["N_t"].values, N_snapshot.values)   # d no fue modificado
assert esc.anio.min() == 2025
print("OK 4) cohortes <=2024 sin cambios (N_t y TBP) y `d` intacto")

### 9.9 Lectura y límites

* **Qué se puede decir:** la serie actual (N de la ONU) **sobrestima** los nacimientos de las cohortes 2025+ respecto de lo que muestran RENAPER y la TGF observada; incluso con una recuperación lenta hasta el valor INDEC 2035, las cohortes 2025–2030 quedan por debajo de lo que supone la ONU. Con persistencia o caída, $TBP_C$ de las cohortes recientes cae en forma sostenida frente a la serie actual.
* **Qué NO se puede decir:** que estos escenarios sean pronósticos; que 390 mil sea el dato oficial 2025 (es una estimación sobre un dato provisorio); que el nivel absoluto del $TBP$ de las cohortes ≥2025 sea definitivo (**$J$ no está re-ponderado**: sección 10).
* **Supuestos a validar por el grupo:** (i) estructura de edades de la fecundidad constante (la proporcionalidad TGF × mujeres 15–49); (ii) ventana 2022–2024 para la relación RENAPER/DEIS; (iii) los tres valores de TGF (1,04 / 1,32 / 0,95 en 2028).
* **Pendiente:** reemplazar el 2025 provisorio por el definitivo cuando el RENAPER lo publique.